# دمج LoRA مع Gemma 4 E4B — النسخة المصححة (يحافظ على الوسائط المتعددة)

**المشاكل اللي انصلحت من الدفتر الأصلي:**

| # | المشكلة | الأثر | الحل |
|---|---|---|---|
| 1 | التحميل بـ `AutoModelForCausalLM` | يحمّل الجزء اللغوي فقط — أوزان الرؤية/الصوت تنحذف من النموذج المحفوظ | `AutoModelForImageTextToText` |
| 2 | حفظ `AutoTokenizer` فقط | ضياع `preprocessor_config.json` و `processor_config.json` → النموذج ما يستقبل صور/صوت حتى لو الأوزان سليمة | حفظ `AutoProcessor` (يشمل الـ tokenizer + كل ملفات المعالجة) |
| 3 | خلية تحميل أولى بدون `torch_dtype` | تحميل بـ float32 = ضعف الذاكرة + خطر OOM على A100 40GB | خلية محذوفة (مكررة أصلاً) |
| 4 | `hf_hub_download` بدون token بخلية التحقق (الخطوة 7) | Gemma مستودع مقيّد (gated) → فشل صامت بجلب ملفات المعالج | تمرير الـ token |
| 5 | فحص المفاتيح يقارن مع base محمّل بكلاس ناقص | التحقق يعطي ✅ كاذب لأن المرجع نفسه ناقص | المرجع الآن الكلاس الكامل + فحص صريح لوجود مفاتيح `vision_tower` و `audio_tower` |
| 6 | `push_to_hub` للـ tokenizer بس | نفس مشكلة رقم 2 على الريبو | رفع مجلد كامل يحتوي ملفات الـ processor |

**القاعدة الذهبية:** الـ LoRA عندك مقيّد بالـ regex على طبقات اللغة فقط، فالدمج ما يمس أبراج الرؤية/الصوت — الخطر كله بمرحلة **التحميل والحفظ**، مو بالأوزان.

In [15]:
# 1) تسجيل الدخول
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('ameerwisam2005')
login(token=hf_token)

In [16]:
# 2) الدمج الصحيح — بالكلاس الكامل متعدد الوسائط
!pip install --upgrade torchao

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor  # ✅ مو AutoModelForCausalLM
from peft import PeftModel

base_model_id = "google/gemma-4-E4B-it"
adapter_model_id = "ameer4wisam/gemma-iraqi-finetune"
save_dir = "./gemma-iraqi-merged"

# ✅ الكلاس الكامل يحمّل نموذج اللغة + برج الرؤية + برج الصوت
base_model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    dtype=torch.bfloat16,   # ✅ تم تصحيح torch_dtype إلى dtype
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_model_id)
merged_model = model.merge_and_unload()

merged_model.save_pretrained(save_dir, safe_serialization=True)

# ✅ AutoProcessor يحفظ: tokenizer + preprocessor_config.json + processor_config.json
#    + special_tokens_map.json + chat_template — كلها بأمر واحد
processor = AutoProcessor.from_pretrained(base_model_id)
processor.save_pretrained(save_dir)

print("✅ تم الدمج والحفظ بالكلاس الكامل")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ تم الدمج والحفظ بالكلاس الكامل


In [17]:
enc = model_check.config  # النموذج محمّل، جرّب توليد حتمي
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(save_dir)
e = tok.apply_chat_template(
    [{"role": "user", "content": "شكد سعر المكيف؟"}],
    add_generation_prompt=True, return_tensors="pt", return_dict=True)
out = model_check.generate(
    input_ids=e["input_ids"].to(model_check.device),
    attention_mask=e["attention_mask"].to(model_check.device),
    max_new_tokens=64, do_sample=False)
print(tok.decode(out[0][e["input_ids"].shape[1]:], skip_special_tokens=True))

سعره 150,000 دينار، وله ضمان سنة.


In [18]:
# المقارنة الصحيحة: شنو عند Google وشنو عندك
import os
from huggingface_hub import list_repo_files, hf_hub_download
import shutil

base_repo = "google/gemma-4-E4B-it"
remote_files = list_repo_files(base_repo, token=hf_token)

# الملفات غير الوزنية اللي تهمنا (إعدادات + معالج + توكنايزر)
config_files = [f for f in remote_files
                if f.endswith((".json", ".jinja", ".model", ".txt"))
                and not f.startswith(".")
                and "index" not in f]          # index الأوزان ماله علاقة، أوزانك مختلفة

print("ملفات الإعدادات بالريبو الأساسي:")
missing = []
for f in config_files:
    exists = os.path.exists(os.path.join(save_dir, f))
    print(("✅" if exists else "❌"), f)
    if not exists:
        missing.append(f)

# نسخ الناقص فقط
for f in missing:
    p = hf_hub_download(repo_id=base_repo, filename=f, token=hf_token)
    shutil.copy(p, os.path.join(save_dir, f))
    print(f"📥 نُسخ: {f}")

print("\n✅ مجلدك هسه مطابق للريبو الأساسي بملفات الإعدادات")

ملفات الإعدادات بالريبو الأساسي:
✅ chat_template.jinja
✅ config.json
✅ generation_config.json
✅ processor_config.json
✅ tokenizer.json
✅ tokenizer_config.json

✅ مجلدك هسه مطابق للريبو الأساسي بملفات الإعدادات


In [19]:
from transformers import AutoProcessor
proc = AutoProcessor.from_pretrained(save_dir)
print(type(proc).__name__)
print("image_processor:", type(getattr(proc, "image_processor", None)).__name__)

Gemma4Processor
image_processor: Gemma4ImageProcessor


In [25]:
# 1) فحص الريبو المرفوع (ثواني، بدون أوزان)
from transformers import AutoConfig, AutoProcessor

repo = "ameer4wisam/gemma-iraqi-finetune-v2"
cfg = AutoConfig.from_pretrained(repo)
proc = AutoProcessor.from_pretrained(repo)

print("الإعدادات:", type(cfg).__name__)
print("المعالج:", type(proc).__name__)
print("معالج الصور:", type(getattr(proc, "image_processor", None)).__name__)

الإعدادات: Gemma4Config
المعالج: Gemma4Processor
معالج الصور: Gemma4ImageProcessor


In [26]:
# 2) الدليل النهائي: استدلال بصورة من الريبو مباشرة
import torch, requests
from PIL import Image
from transformers import AutoModelForImageTextToText

model = AutoModelForImageTextToText.from_pretrained(
    repo, torch_dtype=torch.bfloat16,
    device_map="auto", attn_implementation="eager").eval()

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "شنو تشوف بالصورة؟"}]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

شنو هالمخلوق؟ كلب؟


In [32]:
# 4) الرفع المباشر (بدون شروط أو تحقق)
import os
os.environ["HF_HUB_USE_XET"] = "0"  # تجاوز مشاكل Xet التجريبي
import huggingface_hub._commit_api
huggingface_hub._commit_api.is_xet_available = lambda: False

from huggingface_hub import HfApi

target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"
api = HfApi(token=hf_token)
api.create_repo(repo_id=target_repo, exist_ok=True)

# رفع المجلد بالكامل
api.upload_folder(
    folder_path=save_dir,
    repo_id=target_repo,
    repo_type="model",
    commit_message="Forced Upload: Merged LoRA with model weights and files",
)
print(f"🎉 تم الرفع إلى {target_repo}")

🎉 تم الرفع إلى ameer4wisam/gemma-iraqi-finetune-v2


### 5) فحص ما بعد الرفع — تأكد الوسائط رجعت

هذا الفحص يحسم إذا المشكلة انحلت: نحمّل من الريبو مباشرة ونتأكد من نوع المعالج ووجود إعدادات الرؤية.

In [33]:
# 5) فحص سريع من الريبو (بدون تحميل الأوزان — ثواني فقط)
from transformers import AutoProcessor, AutoConfig

target_repo = "ameer4wisam/gemma-iraqi-finetune-v2"

cfg = AutoConfig.from_pretrained(target_repo)
proc = AutoProcessor.from_pretrained(target_repo)

print(f"نوع الإعدادات: {type(cfg).__name__}")
print(f"نوع المعالج:  {type(proc).__name__}")

checks = {
    "vision_config موجود": hasattr(cfg, "vision_config") and cfg.vision_config is not None,
    "audio_config موجود": hasattr(cfg, "audio_config") and cfg.audio_config is not None,
    "المعالج كامل (مو مجرد tokenizer)": "Processor" in type(proc).__name__,
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

if all(checks.values()):
    print("\n🎉 الوسائط المتعددة سليمة بالنموذج المرفوع")
else:
    print("\n🛑 بعدها ناقصة — أعد خلية الدمج (2) والتحقق (3)")

tokenizer_config.json:   0%|          | 0.00/2.74k [00:00<?, ?B/s]

نوع الإعدادات: Gemma4Config
نوع المعالج:  Gemma4Processor
✅ vision_config موجود
✅ audio_config موجود
✅ المعالج كامل (مو مجرد tokenizer)

🎉 الوسائط المتعددة سليمة بالنموذج المرفوع


In [35]:
# 6) اختبار استدلال نصي (وصفة الاستدلال المعتمدة: حتمي، eager، 64 توكن)
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "ameer4wisam/gemma-iraqi-finetune-v2"

processor = AutoProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",   # مطلوب لمعمارية E4B
).eval()

# اكتشاف توكن التوقف تلقائياً (لا تكتبه يدوياً)
_probe_enc = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt")
_probe = _probe_enc["input_ids"][0].tolist()

special = set(tokenizer.all_special_ids)
stop_ids = list({int(t) for t in _probe[-3:] if int(t) in special}
                | {tokenizer.eos_token_id})

messages = [
    {"role": "system", "content": "أنت بائع عراقي محترف بمحل أجهزة كهربائية. جاوب باللهجة العراقية، قصير ومباشر."},
    {"role": "user", "content": "هلو، عندكم مكيفات سبلت؟"},
]
enc = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    return_tensors="pt", return_dict=True)
with torch.no_grad():
    out = model.generate(
        input_ids=enc["input_ids"].to(model.device),
        attention_mask=enc["attention_mask"].to(model.device),
        eos_token_id=stop_ids, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

هلا بيك، تبيع ايفيكشنال أحسن بـ711,000 دينار، مريح ووافر


In [36]:
# 7) اختبار وسائط (صورة) — الدليل النهائي إن الوسائط تشتغل
import requests
from PIL import Image

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(url, stream=True).raw)

messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "شنو تشوف بالصورة؟"},
    ],
}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

شنو هالمخلوق؟ كلب؟


In [40]:
# ============================================================
#  خلية الاستدلال الاحترافية — gemma-iraqi-finetune-v2
#  الوصفة المعتمدة: حتمي | eager | 64 توكن | كتالوج محقون | فحص أرقام
# ============================================================
import re
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "ameer4wisam/gemma-iraqi-finetune-v2"   # أو "./gemma-iraqi-merged" محلياً

# ---------- 1) التحميل (كلاس كامل متعدد الوسائط + تقرير تحميل) ----------
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

model, _info = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",        # إلزامي لمعمارية E4B
    output_loading_info=True,           # فحص السلامة المعتمد
)
model.eval()
assert not _info["missing_keys"],    f"🛑 أوزان ناقصة: {_info['missing_keys'][:5]}"
assert not _info["unexpected_keys"], f"🛑 أوزان غريبة (بقايا LoRA?): {_info['unexpected_keys'][:5]}"
print(f"✅ النموذج سليم | المعالج: {type(processor).__name__}")

# ---------- 2) اكتشاف توكنات التوقف تلقائياً (لا تكتبها يدوياً) ----------
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False, return_dict=True, return_tensors="pt",
)["input_ids"][0].tolist()
_special = set(tokenizer.all_special_ids)
STOP_IDS = list({t for t in _probe[-3:] if t in _special} | {tokenizer.eos_token_id})
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else STOP_IDS[0]
print(f"✅ توكنات التوقف: {STOP_IDS}")

# ---------- 3) الكتالوج + System Prompt (الحقائق هنا، مو بالأوزان) ----------
CATALOG = """المنتجات المتوفرة حالياً (التزم بيها حرفياً):

الماركات المتوفرة: LG فقط (إذا سأل الزبون عن الماركات، گله عدنا LG بس هسه)

- مكيف LG سبلت طن واحد: 700,000 دينار، ضمان سنتين، استهلاك ~1.0 كيلوواط/ساعة
- مكيف LG سبلت طن ونص: 950,000 دينار، ضمان سنتين، استهلاك ~1.5 كيلوواط/ساعة
  (الفرق بالكهرباء: الطن ونص يصرف حوالي 50% أكثر من الطن الواحد)

- التركيب: 50,000 دينار للجهاز الواحد
- التوصيل داخل بغداد: مجاني | خارج بغداد: 20,000 دينار

خصم شراء قطعتين 5% (المجاميع محسوبة مسبقاً - لا تحسب غيرها):
- 2 × طن واحد: 1,400,000 قبل الخصم -> 1,330,000 بعد الخصم
- 2 × طن ونص: 1,900,000 قبل الخصم -> 1,805,000 بعد الخصم"""

SYSTEM_PROMPT = f"""أنت بائع عراقي محترف بمحل أجهزة كهربائية.

{CATALOG}

قواعد صارمة:
1. جاوب باللهجة العراقية الأصيلة فقط: شنو، شلون، هسه، اكو، ماكو، زين، خوش، شكد.
2. جاوب قصير ومباشر، جملة أو جملتين مثل البائع الحقيقي.
3. الأسعار والأرقام من القائمة أعلاه فقط - لا تخترع أي رقم ولا تحسب مجاميع جديدة.
4. جاوب على السؤال المطلوب بالضبط - لا تكرر معلومات ما انسألت عنها.
5. إذا الزبون گال يريد "اثنين" بدون تحديد الموديل، اسأله: "اثنين طن واحد لو طن ونص؟"
6. لا تدّعي مخزون أو ندرة أو عروض غير مكتوبة بالقائمة.
7. إذا طلب منتج غير موجود، گله: "والله هذا ماكو عدنا هسه" واعرض البديل إذا مناسب.
8. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
9. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب."""

# ---------- 4) حارس الأرقام (regression guard) ----------
_ALLOWED = set(re.findall(r"\d[\d,\.]*", CATALOG))

def check_numbers(reply: str) -> list:
    """أي رقم مالي بالرد لازم يكون موجود حرفياً بالكتالوج."""
    bad = []
    for num in re.findall(r"\d[\d,\.]*", reply):
        clean = num.rstrip(".,")
        if clean in _ALLOWED:
            continue
        if "," not in clean and "." not in clean and len(clean) <= 2:
            continue  # عدد قطع/سنين، مو سعر
        bad.append(clean)
    return bad

# ---------- 5) دالة الاستدلال الموحدة (نص + صورة اختيارية) ----------
class IraqiSalesBot:
    """محادثة متعددة الأدوار: حتمي دائماً، كتالوج محقون، فحص أرقام لكل رد."""

    def __init__(self, system_prompt: str = SYSTEM_PROMPT):
        self.messages = [{"role": "system", "content": system_prompt}] if system_prompt else []

    def _generate(self, inputs) -> str:
        with torch.no_grad():
            out = model.generate(
                **inputs,
                eos_token_id=STOP_IDS,
                pad_token_id=PAD_ID,
                max_new_tokens=64,       # أجوبة التدريب قصيرة؛ الطول الزائد = هذيان
                do_sample=False,          # حتمي فقط — الوضع المتوازن محذوف نهائياً
            )
        return tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()

    def chat(self, user_text: str, image=None, verbose: bool = True) -> str:
        # بناء محتوى الرسالة (نص، أو نص + صورة)
        if image is not None:
            content = [{"type": "image", "image": image},
                       {"type": "text", "text": user_text}]
        else:
            content = user_text
        self.messages.append({"role": "user", "content": content})

        # الترميز: processor للوسائط، tokenizer للنص الصرف
        if image is not None:
            inputs = processor.apply_chat_template(
                self.messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt",
            ).to(model.device, dtype=torch.bfloat16)
        else:
            enc = tokenizer.apply_chat_template(
                self.messages, add_generation_prompt=True,
                return_dict=True, return_tensors="pt",
            )
            inputs = {k: v.to(model.device) for k, v in enc.items()}

        reply = self._generate(inputs)

        # نسجل بالسجل نسخة نصية (حتى ما نعيد إرسال الصورة بكل دور)
        self.messages[-1] = {"role": "user", "content": user_text}
        self.messages.append({"role": "assistant", "content": reply})

        # حارس الأرقام
        bad = check_numbers(reply)
        if verbose:
            ctx = len(tokenizer.apply_chat_template(self.messages, add_generation_prompt=False))
            print(f"👤 {user_text}")
            print(f"🤖 {reply}   (سياق: {ctx} توكن)")
            if bad:
                print(f"   🚨 أرقام مو بالكتالوج: {bad}")
            print("-" * 60)
        return reply

    def reset(self):
        self.messages = self.messages[:1]  # يحتفظ بالـ system prompt فقط

# ---------- 6) الاستخدام ----------
bot = IraqiSalesBot()
bot.chat("هلو، عندكم مكيفات سبلت؟")
bot.chat("شكد سعر الطن الواحد؟")
bot.chat("وإذا أخذت اثنين طن ونص؟")

# مثال ويا صورة (اختياري):
# from PIL import Image
# img = Image.open("device.jpg")
# bot.chat("عندكم مثل هذا المكيف؟", image=img)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

✅ النموذج سليم | المعالج: Gemma4Processor
✅ توكنات التوقف: [1, 106]
👤 هلو، عندكم مكيفات سبلت؟
🤖 عدنا طن واحد لو طن ونص، شتفضل؟   (سياق: 2 توكن)
------------------------------------------------------------
👤 شكد سعر الطن الواحد؟
🤖 مكيف LG سبلت طن واحد متوفر عدنا بـ700,000 دينار وعليه ضمان سنتين   (سياق: 2 توكن)
------------------------------------------------------------
👤 وإذا أخذت اثنين طن ونص؟
🤖 ماشي، تشتري اثنين طن ونص؟ أحسبلك خصم 5%   (سياق: 2 توكن)
------------------------------------------------------------


'ماشي، تشتري اثنين طن ونص؟ أحسبلك خصم 5%'